# VGG16 — Crop Leaf Disease Detection
**Team 179 | B.Tech Research Project — CNN Baseline**

This notebook fine-tunes **VGG16** on the combined **PlantDoc + PlantVillage** dataset for a fair CNN vs Transformer comparison.

### Why VGG16?
- Simple, well-understood architecture — 16 weight layers, all 3×3 conv filters
- CNN processes image through LOCAL convolutional filters — contrasts clearly with ViT/Swin global attention
- Pretrained on ImageNet-1k — strong transfer learning baseline
- Easy to explain mathematically: **Output = ReLU(W * x + b)** at each conv layer

### VGG16 Architecture
- 13 convolutional layers (all 3×3, stride 1)
- 5 max-pooling layers
- 3 fully connected layers
- 138M parameters total
- Input: 224×224×3

### CNN vs Transformer — the key difference
- VGG16 uses LOCAL 3×3 filters — each filter only sees a tiny patch
- ViT/Swin use GLOBAL attention — every patch sees every other patch
- This is exactly what your paper compares

### Dataset
- PlantDoc + PlantVillage combined (~56,000 images, 30 classes)
- Same dataset as ViT v2 and Swin-Base — fair comparison

### Hardware
- Kaggle T4 GPU (16GB VRAM)
- Estimated training time: **1.5–2.5 hours**


## Step 1: Install Dependencies

In [ ]:
!pip install scikit-learn matplotlib seaborn --quiet

import torch
import torchvision
print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
import cv2
cv2.setNumThreads(0)

## Step 2: Load PlantDoc + PlantVillage

> **Add both datasets via + Add Data:**
> 1. Search **`plant-doc-dataset`** (abdulhasibuddin) → Add
> 2. Search **`plantvillage-dataset`** (abdallahalidev) → Add
>
> This is identical to the transformer notebooks — ensures a fair CNN vs Transformer comparison.

In [ ]:
import os
from pathlib import Path
from collections import Counter
import json
import random
# Add this at the very top of your notebook (Step 1 or 2)
import torch.multiprocessing as mp

# Force spawn method for workers
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass # Ignore if it's already set

PLANTDOC_BASE     = None
PLANTVILLAGE_BASE = None

for p in Path('/kaggle/input').iterdir():
    name = p.name.lower()
    if 'plantvillage' in name or 'plant-village' in name or 'plant_village' in name:
        PLANTVILLAGE_BASE = p
    elif 'plantdoc' in name or 'plant-doc' in name or 'plant_doc' in name:
        PLANTDOC_BASE = p

print('Datasets found in /kaggle/input:')
for p in Path('/kaggle/input').iterdir():
    print(f'  {p.name}')

if PLANTDOC_BASE is None:
    raise FileNotFoundError('PlantDoc not found — add abdulhasibuddin/plant-doc-dataset')
if PLANTVILLAGE_BASE is None:
    raise FileNotFoundError('PlantVillage not found — add abdallahalidev/plantvillage-dataset')

print(f'\nPlantDoc     : {PLANTDOC_BASE}')
print(f'PlantVillage : {PLANTVILLAGE_BASE}')

In [ ]:
# PlantVillage → PlantDoc class name mapping
# Identical to ViT v2 notebook for consistency
PV_TO_PLANTDOC = {
    'Tomato___Early_blight':                  'Tomato Early blight leaf',
    'Tomato___Late_blight':                   'Tomato leaf late blight',
    'Tomato___Bacterial_spot':                'Tomato leaf bacterial spot',
    'Tomato___Septoria_leaf_spot':            'Tomato Septoria leaf spot',
    'Tomato___Leaf_Mold':                     'Tomato mold leaf',
    'Tomato___Tomato_mosaic_virus':           'Tomato leaf mosaic virus',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'Tomato leaf yellow virus',
    'Tomato___healthy':                       'Tomato leaf',
    'Potato___Early_Blight':                  'Potato leaf early blight',
    'Potato___Late_Blight':                   'Potato leaf late blight',
    'Apple___Apple_scab':                     'Apple Scab Leaf',
    'Apple___healthy':                        'Apple leaf',
    'Apple___Cedar_apple_rust':               'Apple rust leaf',
    'Grape___healthy':                        'grape leaf',
    'Grape___Black_rot':                      'grape leaf black rot',
    'Corn_(maize)___Gray_leaf_spot':          'Corn Gray leaf spot',
    'Corn_(maize)___Northern_Leaf_Blight':    'Corn leaf blight',
    'Corn_(maize)___Common_rust_':            'Corn rust leaf',
    'Pepper,_bell___healthy':                 'Bell_pepper leaf',
    'Pepper,_bell___Bacterial_spot':          'Bell_pepper leaf spot',
    'Strawberry___healthy':                   'Strawberry leaf',
    'Raspberry___healthy':                    'Raspberry leaf',
    'Blueberry___healthy':                    'Blueberry leaf',
    'Soybean___healthy':                      'Soyabean leaf',
    'Squash___Powdery_mildew':                'Squash Powdery mildew leaf',
    'Peach___healthy':                        'Peach leaf',
    'Cherry_(including_sour)___healthy':      'Cherry leaf',
}

def load_plantdoc(base):
    samples = []
    base = Path(base)
    split_found = False
    for split in ['train', 'test', 'val', 'valid']:
        split_dir = base / split
        if split_dir.exists():
            split_found = True
            for class_dir in sorted(split_dir.iterdir()):
                if class_dir.is_dir():
                    for img in list(class_dir.glob('**/*.jpg')) + list(class_dir.glob('**/*.png')):
                        samples.append((str(img), class_dir.name))
    if not split_found:
        for class_dir in sorted(base.iterdir()):
            if class_dir.is_dir():
                for img in list(class_dir.glob('**/*.jpg')) + list(class_dir.glob('**/*.png')):
                    samples.append((str(img), class_dir.name))
    return samples

def load_plantvillage(base, name_map):
    samples = []
    base = Path(base)
    search_root = base
    for sub in ['color', 'train', 'plantvillage']:
        if (base / sub).exists():
            search_root = base / sub
            break
    for class_dir in sorted(search_root.iterdir()):
        if class_dir.is_dir() and class_dir.name in name_map:
            mapped = name_map[class_dir.name]
            for img in list(class_dir.glob('**/*.jpg')) + list(class_dir.glob('**/*.png')):
                samples.append((str(img), mapped))
    return samples

print('Loading datasets...')
pd_samples = load_plantdoc(PLANTDOC_BASE)
pv_samples = load_plantvillage(PLANTVILLAGE_BASE, PV_TO_PLANTDOC)
all_samples = pd_samples + pv_samples

class_names  = sorted(list(set([s[1] for s in all_samples])))
class_to_idx = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES  = len(class_names)

print(f'PlantDoc     : {len(pd_samples):,} images')
print(f'PlantVillage : {len(pv_samples):,} images')
print(f'Total        : {len(all_samples):,} images')
print(f'Classes      : {NUM_CLASSES}')

with open('/kaggle/working/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
print('Class names saved!')

## Step 3: Prepare Dataset for VGG16

VGG16 expects:
- Images resized to **224×224**
- Normalized with ImageNet mean/std (same as ViT/Swin for fair comparison)
- **Note:** Unlike transformers, VGG16 applies local 3×3 convolutions — no patch tokenization needed

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from PIL import Image
import torch

# Augmentation for training
# CNNs benefit from standard augmentation — no need for TrivialAugmentWide (that's transformer-specific)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)) # <-- ADD THIS
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PlantDiseaseDataset(Dataset):
    def __init__(self, samples, class_to_idx, transform=None):
        self.samples     = samples
        self.class_to_idx = class_to_idx
        self.transform   = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.class_to_idx[label]

# Stratified 80/10/10 split — same strategy as transformer notebooks
all_labels = [s[1] for s in all_samples]
train_samples, temp_samples, _, temp_labels = train_test_split(
    all_samples, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)
val_samples, test_samples = train_test_split(
    temp_samples, test_size=0.5, stratify=temp_labels, random_state=42
)
print(f'Train: {len(train_samples):,} | Val: {len(val_samples):,} | Test: {len(test_samples):,}')

train_dataset = PlantDiseaseDataset(train_samples, class_to_idx, train_transform)
val_dataset   = PlantDiseaseDataset(val_samples,   class_to_idx, val_transform)
test_dataset  = PlantDiseaseDataset(test_samples,  class_to_idx, val_transform)

# VGG16 is memory-heavy — batch size 32 is safe on T4
import os

# Dynamically find the safe maximum number of workers
# os.cpu_count() returns 4 on a Kaggle T4 instance. 
# We cap it at 4 to prevent I/O bottlenecking on the Kaggle file system.
num_workers = min(4, os.cpu_count() or 1) 

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=num_workers, pin_memory=True , prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=num_workers, pin_memory=True)

print(f'Using {num_workers} workers for DataLoader')

print(f'DataLoaders ready! Train batches: {len(train_loader)}')

## Step 4: Load Pretrained VGG16

**VGG16 architecture breakdown:**
```
Input (224×224×3)
  → Block 1: 2× Conv(64, 3×3) + MaxPool  → 112×112×64
  → Block 2: 2× Conv(128, 3×3) + MaxPool → 56×56×128
  → Block 3: 3× Conv(256, 3×3) + MaxPool → 28×28×256
  → Block 4: 3× Conv(512, 3×3) + MaxPool → 14×14×512
  → Block 5: 3× Conv(512, 3×3) + MaxPool → 7×7×512
  → Flatten → FC(4096) → FC(4096) → FC(NUM_CLASSES)
```
We replace the final FC layer with our disease classes.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load pretrained VGG16
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# Replace final classifier layer for our classes
# Original: Linear(4096, 1000) → New: Linear(4096, NUM_CLASSES)
model.classifier[6] = nn.Linear(4096, NUM_CLASSES)

# Multi-GPU support
model = model.to(device)
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Output classes      : {NUM_CLASSES}')
print('\nVGG16 loaded successfully!')

## Step 5: Train VGG16

**Training strategy for CNN vs Transformer comparison:**
- Optimizer: SGD with momentum (classic CNN optimizer — different from AdamW used for transformers)
- LR: 0.001 with cosine annealing decay
- Class weights: same balanced weighting as transformer notebooks
- Early stopping: patience 7
- Mixed precision FP16 for speed

In [ ]:
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import time
import json

# ─── Hyperparameters ──────────────────────────────────────────────────────────
EPOCHS       = 20
LR           = 1e-3      # Higher LR than transformers — CNNs are more stable
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-4
PATIENCE     = 7

# ─── Class weights for imbalanced dataset ─────────────────────────────────────
train_labels  = [class_to_idx[s[1]] for s in train_samples]
weights       = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f'Class weights computed for {len(weights)} classes')

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# SGD with momentum — standard for CNN training
# In Step 5, replace your optimizer definition with this:
base_params = [p for name, p in model.named_parameters() if 'classifier.6' not in name]
classifier_params = model.module.classifier[6].parameters() if isinstance(model, nn.DataParallel) else model.classifier[6].parameters()

optimizer = SGD([
    {'params': base_params, 'lr': LR * 0.1},      # 1e-4 for pretrained layers
    {'params': classifier_params, 'lr': LR}       # 1e-3 for new classifier layer
], momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

# 👇 ADD THESE TWO LINES 👇
scaler = GradScaler()
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ─── Training Loop ────────────────────────────────────────────────────────────
best_val_acc = 0.0
patience_ctr = 0
history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('Starting VGG16 training...')
print(f'Epochs: {EPOCHS} | LR: {LR} | Batch: 32 | Device: {device}')
print('='*65)

for epoch in range(EPOCHS):
    t0 = time.time()

    # ── TRAIN ──
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss    += loss.item()
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

        if (batch_idx + 1) % 50 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | '
                  f'Loss: {loss.item():.4f}', end='\r')

    # ── VALIDATE ──
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)
            val_loss    += loss.item()
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    t_loss  = train_loss / len(train_loader)
    t_acc   = train_correct / train_total * 100
    v_loss  = val_loss / len(val_loader)
    v_acc   = val_correct / val_total * 100
    elapsed = time.time() - t0

    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)

    print(f'Epoch [{epoch+1:02d}/{EPOCHS}] '
          f'Train Loss: {t_loss:.4f} Acc: {t_acc:.2f}% | '
          f'Val Loss: {v_loss:.4f} Acc: {v_acc:.2f}% | '
          f'LR: {scheduler.get_last_lr()[0]:.2e} | Time: {elapsed:.0f}s')

    scheduler.step()

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        patience_ctr = 0
        
        # Strip DataParallel wrapper before saving to ensure compatibility
        model_to_save = model.module if isinstance(model, nn.DataParallel) else model
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_to_save.state_dict(), # <-- Updated this line
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': v_acc,
            'class_names': class_names
        }, '/kaggle/working/vgg16_best.pth')
        print(f'  ✓ Best model saved! Val Acc: {v_acc:.2f}%')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

print(f'\nTraining complete! Best Val Accuracy: {best_val_acc:.2f}%')
with open('/kaggle/working/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

## Step 6: Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import json

with open('/kaggle/working/training_history.json') as f:
    history = json.load(f)

epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VGG16 Fine-tuning — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_ran, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs_ran, history['val_loss'],   'r-o', label='Val Loss',   markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], 'b-o', label='Train Acc', markersize=4)
axes[1].plot(epochs_ran, history['val_acc'],   'r-o', label='Val Acc',   markersize=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Curve'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved!')

## Step 7: Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import time

# Load best weights
checkpoint = torch.load('/kaggle/working/vgg16_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Best model loaded from epoch {checkpoint["epoch"]+1}')
print(f'Best val accuracy: {checkpoint["val_acc"]:.2f}%')

all_preds, all_labels_list = [], []
test_correct, test_total   = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds   = outputs.argmax(dim=1)
        test_correct += (preds == labels).sum().item()
        test_total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

test_acc = test_correct / test_total * 100
print(f'\n===== TEST SET RESULTS =====')
print(f'Test Accuracy: {test_acc:.2f}%')
print(f'\nPer-class Report:')
print(classification_report(all_labels_list, all_preds, target_names=class_names, zero_division=0))

In [ ]:
import time
import numpy as np
import torch
from torch.cuda.amp import autocast

print(f'\n===== INFERENCE BENCHMARK (VGG16) =====')
model.eval()

# --- 1. LATENCY BENCHMARK (Batch Size 1) ---
# Simulates edge-device real-time processing
dummy_single = torch.zeros(1, 3, 224, 224).to(device)

# Warmup
with torch.no_grad(), autocast():
    for _ in range(20): _ = model(dummy_single)

times_single = []
with torch.no_grad(), autocast():
    for _ in range(200):
        t0 = time.perf_counter()
        _ = model(dummy_single)
        torch.cuda.synchronize()
        times_single.append(time.perf_counter() - t0)

avg_latency = np.mean(times_single) * 1000  # in milliseconds
fps_bs1 = 1.0 / np.mean(times_single)

print(f'[Batch Size 1]  Latency: {avg_latency:.2f} ms | FPS: {fps_bs1:.1f}')

# --- 2. THROUGHPUT BENCHMARK (Batch Size 32) ---
# Simulates cloud/server bulk processing
BATCH_SIZE = 32
dummy_batch = torch.zeros(BATCH_SIZE, 3, 224, 224).to(device)

# Warmup
with torch.no_grad(), autocast():
    for _ in range(20): _ = model(dummy_batch)

times_batch = []
with torch.no_grad(), autocast():
    for _ in range(50): # Fewer iterations needed for large batches
        t0 = time.perf_counter()
        _ = model(dummy_batch)
        torch.cuda.synchronize()
        times_batch.append(time.perf_counter() - t0)

# Total time divided by batch size gives time per image
time_per_batch = np.mean(times_batch)
throughput_fps = BATCH_SIZE / time_per_batch

print(f'[Batch Size 32] Throughput FPS: {throughput_fps:.1f}')

print(f'\n--- Paper-ready result ---')
print(f'VGG16 | Test Acc: {test_acc:.2f}% | Latency (BS1): {avg_latency:.2f}ms | Throughput (BS32): {throughput_fps:.1f} FPS')

In [ ]:
# Confusion matrix — top 15 classes
from collections import Counter

top_idx  = [i for i, _ in Counter(all_labels_list).most_common(15)]
top_names = [class_names[i][:20] for i in top_idx]
remap    = {old: new for new, old in enumerate(top_idx)}

f_preds  = [remap.get(p, 0) for p, l in zip(all_preds, all_labels_list) if l in top_idx]
f_labels = [remap[l] for l in all_labels_list if l in top_idx]

cm      = confusion_matrix(f_labels, f_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=top_names, yticklabels=top_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('VGG16 — Confusion Matrix (Top 15 Classes)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved!')

## Step 8: Grad-CAM Visualization

**Grad-CAM is the CNN equivalent of ViT attention maps.**
Instead of attention weights, it uses gradients flowing back through the last conv layer to show WHAT the CNN was looking at.
This is your paper's visual comparison figure — ViT attention maps vs VGG16 Grad-CAM side by side.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random
import cv2

class GradCAM:
    """Grad-CAM for VGG16 — highlights what the CNN focused on."""
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx):
        self.model.eval()
        output = self.model(input_tensor)
        self.model.zero_grad()
        output[0, class_idx].backward()

        weights  = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam      = (weights * self.activations).sum(dim=1, keepdim=True)
        cam      = torch.relu(cam)
        cam      = cam.squeeze().cpu().numpy()
        cam      = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

# Target: last conv layer of VGG16 (features[28])
base_model = model.module if isinstance(model, torch.nn.DataParallel) else model
gradcam    = GradCAM(base_model, base_model.features[28])

# Visualize on 6 test samples
random.seed(42)
sample_indices = random.sample(range(len(test_dataset)), min(6, len(test_dataset)))

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
fig.suptitle('VGG16 Grad-CAM — What the CNN Focuses On', fontsize=14, fontweight='bold')

for col, idx in enumerate(sample_indices):
    image_tensor, label_idx = test_dataset[idx]
    label    = class_names[label_idx]
    orig_img = Image.open(test_samples[idx][0]).convert('RGB').resize((224, 224))

    inp = image_tensor.unsqueeze(0).to(device).requires_grad_(True)
    cam = gradcam.generate(inp, label_idx)

    with torch.no_grad():
        logits    = base_model(image_tensor.unsqueeze(0).to(device))
        pred_idx  = logits.argmax(dim=1).item()
        pred_label = class_names[pred_idx]
        conf      = torch.softmax(logits, dim=1).max().item()

    correct = '✓' if pred_idx == label_idx else '✗'

    # Row 1: original
    axes[0][col].imshow(orig_img)
    axes[0][col].set_title(f'{label[:18]}', fontsize=7, fontweight='bold')
    axes[0][col].axis('off')

    # Row 2: Grad-CAM overlay
    cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224)))
    cam_norm    = cam_resized / 255.0
    axes[1][col].imshow(orig_img, alpha=0.5)
    axes[1][col].imshow(cam_norm, cmap='jet', alpha=0.6)
    axes[1][col].set_title(f'{correct} {pred_label[:15]}\n{conf:.0%}', fontsize=7)
    axes[1][col].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grad-CAM maps saved!')

## Step 9: Save All Assets

In [ ]:
import json, shutil
from pathlib import Path

save_dir = Path('/kaggle/working/vgg16_final_assets')
save_dir.mkdir(exist_ok=True)

final_metrics = {
    'model':         'VGG16',
    'dataset':       'PlantDoc + PlantVillage (combined)',
    'task':          'Crop Leaf Disease Classification',
    'num_classes':   NUM_CLASSES,
    'test_accuracy': round(test_acc, 2),
    'fps':           round(fps, 1),
    'latency_ms':    round(avg_latency, 2),
    'best_val_acc':  round(best_val_acc, 2),
    'classes':       class_names,
    'architecture': {
        'conv_layers': 13,
        'fc_layers':   3,
        'filter_size': '3x3 (all)',
        'parameters':  '138M'
    },
    'comparison': {
        'ViT_v1':  {'test_acc': 72.27, 'fps': 48.9,  'latency_ms': 20.46},
        'ViT_v2':  {'test_acc': 74.22, 'fps': 70.1,  'latency_ms': 14.26},
        'Swin':    {'test_acc': 73.83, 'fps': 20.5,  'latency_ms': 48.81},
        'VGG16':   {'test_acc': round(test_acc, 2), 'fps': round(fps, 1), 'latency_ms': round(avg_latency, 2)}
    }
}

with open(save_dir / 'metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

for fname in ['vgg16_best.pth', 'training_curves.png', 'confusion_matrix.png',
              'gradcam_maps.png', 'training_history.json', 'class_names.json']:
    src = Path('/kaggle/working') / fname
    if src.exists():
        shutil.copy(src, save_dir / fname)

shutil.make_archive('/kaggle/working/vgg16_assets', 'zip', save_dir)

print('All assets saved!')
print(json.dumps(final_metrics, indent=2))
print('\nDownload vgg16_assets.zip from the Kaggle output panel')

## Notes for your paper

| Aspect | VGG16 (CNN) | ViT v2 (Transformer) | Swin (Transformer) |
|--------|-------------|---------------------|--------------------|
| Receptive field | Local 3×3 patches | Global (all patches) | Local windows, shifted |
| Optimizer | SGD + momentum | AdamW | AdamW |
| Parameters | 138M | 86M | 88M |
| Inductive bias | Strong (translation invariance) | None | Partial (locality) |
| Visualization | Grad-CAM | Attention maps | Attention maps |

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce batch size to 16 |
| Dataset not found | Add both datasets via + Add Data |
| Grad-CAM fails with DataParallel | Already handled — uses `model.module` |
| Low accuracy after 5 epochs | Normal — VGG16 converges slower than transformers on this task |
| Session timeout | Use **Save & Run All** background session |

---
**Team 179 | VGG16 CNN Baseline | B.Tech Research Project**

Expected results:
- Test Accuracy: **72–80%** (CNN usually slightly below transformers on this task)
- FPS: **80–150** (CNNs are faster at inference)
- Latency: **6–12ms**